In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import TruncatedSVD
from pathlib import Path

### Set the random seed

In [2]:
np.random.seed(42)

### Download the data

In [3]:
path = "../data/processed/movies_reindexed.parquet"

df = pd.read_parquet(path)
df.head()

,movie_id,title,genres,item_idx
0,1,Toy Story (1995),Animation|Children's|Comedy,0
1,2,Jumanji (1995),Adventure|Children's|Fantasy,1
2,3,Grumpier Old Men (1995),Comedy|Romance,2
3,4,Waiting to Exhale (1995),Comedy|Drama,3
4,5,Father of the Bride Part II (1995),Comedy,4


### Process the data

In [4]:
years = df['title'].str.split('(').str[-1].str.rstrip(')').astype('Int64')
years.head()

0    1995
1    1995
2    1995
3    1995
4    1995
Name: title, dtype: Int64

In [5]:
title = df['title'].str.split('(').str[0]
title.head()

0                      Toy Story 
1                        Jumanji 
2               Grumpier Old Men 
3              Waiting to Exhale 
4    Father of the Bride Part II 
Name: title, dtype: object

In [6]:
df['years'] = years
df['title'] = title
df['genres'] = df['genres'].str.split('|')

df.head()

,movie_id,title,genres,item_idx,years
0,1,Toy Story,"[Animation, Children's, Comedy]",0,1995
1,2,Jumanji,"[Adventure, Children's, Fantasy]",1,1995
2,3,Grumpier Old Men,"[Comedy, Romance]",2,1995
3,4,Waiting to Exhale,"[Comedy, Drama]",3,1995
4,5,Father of the Bride Part II,[Comedy],4,1995


##### OHE for genres

In [7]:
mlb = MultiLabelBinarizer()

genres = mlb.fit_transform(df['genres'])
all_genres = list(mlb.classes_)

##### TF-IDF for title

In [8]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    max_features=20000
)

X_tfidf = vectorizer.fit_transform(df['title'])
X_tfidf.shape

(3706, 3658)

##### SVD for TF-IDF title

In [9]:
svd = TruncatedSVD(n_components=512)

title_svd = svd.fit_transform(X_tfidf)
title_svd.shape

(3706, 128)

##### MinMaxScaler for years

In [10]:
scaler = MinMaxScaler()

y = df[['years']].to_numpy()
scaled_y = scaler.fit_transform(y)
scaled_y.shape

(3706, 1)

### Save the data

In [11]:

OUT_DIR = Path("../data/processed/item_features")
OUT_DIR.mkdir(parents=False, exist_ok=True)

np.save(OUT_DIR / "title_svd512.npy", title_svd.astype(np.float32))
np.save(OUT_DIR / "genres.npy", genres.astype(np.float32))
np.save(OUT_DIR / "year_minmax.npy", scaled_y.astype(np.float32))

In [12]:
df[["movie_id", "item_idx", "title", "years", "genres"]].to_parquet(
    OUT_DIR / "item_meta.parquet", index=False
)